## Transform Results Data
- Read bronze Results table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(constructorId->constructor_id,driverId->driver_id)
- Rename columns to make them more meaningful(grid->grid_position)
- Filter out rows where seaon,round,constructor_id and driver_id is null (business key validation)
- Remove duplicate records.
- Transform values of column race_name to Title Case
- Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
import pyspark.sql.functions as F

### Step 1 to 7 Read ,trasform and perform data quality checks

In [0]:
results_df=(
    spark.table(bronze_table)
    .drop("url")
    .withColumnsRenamed(
        {"constructorId":"constructor_id",
         "driverId":"driver_id",
         "raceName":"race_name",
         "date":"race_date",
         "time":"race_time",
         "grid":"grid_position",
         "laps":"completed_laps",
         "number":"car_number",
         "position":"final_position",
         "positionText":"final_position_text"
        }
    )
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()
    )
    .dropDuplicates(["season","round","driver_id","constructor_id"])
    .withColumn("race_name",F.initcap(F.col("race_name")))
)


### Step-8 Write the transformed data to silver drivers table

In [0]:
(
   results_df
    .write
    .mode("overwrite")
    .format("delta")
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.read.table(silver_table))